# Acoustic Machine-Fault Detection for Predictive Maintenance Using CNN and CNN-GRU
**Author:** Aditi Mishra (B.Tech Electronics and Computer Science)  
**Dataset:** MIMII (Malfunctioning Industrial Machine Investigation and Inspection) Benchmark  
**Models:** 2D CNN Baseline (`FanCNN`) vs. Spatio-Temporal Hybrid (`FanCNNGRU`)  
**Hardware Environment:** Windows PC (CPU-Only / Local Runtime)

---
### Project Abstract
This capstone project implements an automated, non-contact predictive maintenance system that classifies industrial machine audio (Fans, Pumps, Sliders, Valves) as **Normal (`0`)** or **Abnormal (`1`)** under variable background factory noise (-6 dB, 0 dB, +6 dB).

Input audio recordings are converted into **Log-Mel Spectrograms** of dimension `[128, 157]` (128 Mel frequency bins × 157 time frames). Two neural network architectures are evaluated:
1. **FanCNN:** 2D Convolutional baseline capturing spatial-frequency features.
2. **FanCNNGRU:** Hybrid model combining 2D convolutions with a Recurrent Gated Unit (GRU) to capture temporal acoustic transitions.


In [ ]:
import os
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
import librosa
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, classification_report
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Ensure strict reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PROJECT_DIR = r'C:\Capstone Project files'
print(f'Active Device: {DEVICE}')
print(f'Project Root: {PROJECT_DIR}')


### 1. Environment and Reproducibility Setup
The project operates in a CPU-only environment with PyTorch, Librosa, and Scikit-Learn. Random seeds are locked to `42` to guarantee exact deterministic reproducibility across all data splits and evaluation runs.


In [ ]:
SR = 16000
DURATION = 5.0
TARGET_SAMPLES = int(SR * DURATION)  # 80,000 samples
N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 512
DB_FLOOR = -80.0
DB_CEILING = 0.0

def waveform_to_logmel(audio):
    audio = np.asarray(audio, dtype=np.float32)
    if audio.ndim > 1:
        audio = np.mean(audio, axis=-1)
    if len(audio) < TARGET_SAMPLES:
        audio = np.pad(audio, (0, TARGET_SAMPLES - len(audio)), mode='constant')
    else:
        audio = audio[:TARGET_SAMPLES]

    mel = librosa.feature.melspectrogram(
        y=audio, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS, power=2.0
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)
    log_mel = np.clip(log_mel, DB_FLOOR, DB_CEILING)
    log_mel = (log_mel - DB_FLOOR) / (DB_CEILING - DB_FLOOR)
    return log_mel.astype(np.float32)

print('Feature extraction parameters configured.')
print(f'Input tensor shape per 5s audio clip: [1, 1, {N_MELS}, {int(np.ceil(TARGET_SAMPLES / HOP_LENGTH))}]')


### 2. Acoustic Feature Extraction Pipeline
Raw audio is standardized to mono 16 kHz and 5.0 seconds (80,000 samples). Using Short-Time Fourier Transform (STFT) with a Hann window of 1024 samples and a hop size of 512 samples, the audio is mapped into 128 Mel-frequency bands. 
Logarithmic compression converts energy to decibels relative to the peak power, clipped to $[-80, 0]$ dB and normalized to $[0.0, 1.0]$. The resulting feature matrix has shape `[128, 157]`.


In [ ]:
from src.dataset import load_project_data

X, y, metadata, train_idx, val_idx, test_idx = load_project_data(PROJECT_DIR)

print(f'Total dataset size: {len(X)} clips')
print(f'Feature array shape: {X.shape}')
print(f'Labels array shape: {y.shape}')
print(f'Train split size: {len(train_idx)} (70%)')
print(f'Validation split size: {len(val_idx)} (15%)')
print(f'Test split size: {len(test_idx)} (15%)')
print('\nClass Distribution across Dataset:')
print(pd.Series(y).map({0: 'Normal (0)', 1: 'Abnormal (1)'}).value_counts())


### 3. Dataset Splitting & Split Verification
The dataset consists of 2,400 balanced audio clips (1,200 normal and 1,200 abnormal) spanning 12 configurations (4 machines × 3 noise levels).
- **Train Split:** 1,680 clips (70%)
- **Validation Split:** 360 clips (15%)
- **Test Split:** 360 clips (15%)
Stratification ensures equal 50:50 class balance in every split.


In [ ]:
from src.models import FanCNN, FanCNNGRU, load_model

cnn_ckpt = os.path.join(PROJECT_DIR, 'cnn_improved_best.pt')
gru_ckpt = os.path.join(PROJECT_DIR, 'cnn_gru_improved_best.pt')

cnn = load_model('CNN', cnn_ckpt, device='cpu')
cnn_gru = load_model('CNN-GRU', gru_ckpt, device='cpu')

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'FanCNN Trainable Parameters: {count_parameters(cnn):,}')
print(f'FanCNNGRU Trainable Parameters: {count_parameters(cnn_gru):,}')


### 4. Neural Network Model Architectures
- **`FanCNN` (25,857 parameters):** 3 convolutional blocks with 16, 32, and 64 filters of size 3×3, followed by batch normalization, ReLU activation, and 2×2 max-pooling. An adaptive average pooling layer and 30% dropout connect to a scalar linear output logit.
- **`FanCNNGRU` (51,073 parameters):** Replaces 2D spatial pooling with frequency-axis average pooling to preserve the 1D temporal dimension, followed by a 64-unit Gated Recurrent Unit (GRU) to model sequential acoustic patterns across time frames.


In [ ]:
overall_df = pd.read_csv(os.path.join(PROJECT_DIR, 'outputs', 'metrics', 'overall_model_evaluation.csv'))
test_comparison = overall_df[overall_df['split'] == 'Test'].copy()

display_cols = ['model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'false_alarm_rate_fpr', 'missed_fault_rate_fnr', 'mean_latency_ms']
print('=== Test Set Benchmark Results (Default Threshold = 0.50) ===')
display(test_comparison[display_cols].style.format({
    'accuracy': '{:.2%}', 'precision': '{:.2%}', 'recall': '{:.2%}',
    'f1': '{:.2%}', 'roc_auc': '{:.4f}', 'false_alarm_rate_fpr': '{:.2%}',
    'missed_fault_rate_fnr': '{:.2%}', 'mean_latency_ms': '{:.2f} ms'
}))


### 5. Benchmark Performance Analysis (Threshold = 0.50)
- **CNN Baseline:** Accuracy = **73.61%**, Precision = **76.40%**, Recall = **68.33%**, F1 = **72.14%**, ROC-AUC = **0.8101**, Latency = **2.29 ms**.
- **CNN-GRU Hybrid:** Accuracy = **74.44%**, Precision = **74.44%**, Recall = **74.44%**, F1 = **74.44%**, ROC-AUC = **0.8149**, Latency = **2.35 ms**.

**Key Observation:** CNN-GRU detects **+6.11% more faults** (Recall 74.44% vs 68.33%), cutting the missed-fault rate from 31.67% down to 25.56% with negligible latency overhead (+0.06 ms).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=150)

# 1. Confusion Matrix
from sklearn.metrics import confusion_matrix
sub_X = X[test_idx]
sub_y = y[test_idx]

from src.evaluation import compute_predictions
cnn_probs, _ = compute_predictions(cnn, sub_X, device='cpu')
gru_probs, _ = compute_predictions(cnn_gru, sub_X, device='cpu')

cm_gru = confusion_matrix(sub_y, (gru_probs >= 0.5).astype(int), labels=[0, 1])
axes[0].imshow(cm_gru, cmap='Blues')
axes[0].set_title('CNN-GRU Test Confusion Matrix (T=0.50)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')
axes[0].set_xticks([0, 1], ['Normal (0)', 'Abnormal (1)'])
axes[0].set_yticks([0, 1], ['Normal (0)', 'Abnormal (1)'])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, f'{cm_gru[i, j]}\n({cm_gru[i, j]/len(sub_y)*100:.1f}%)',
                     ha='center', va='center', color='white' if cm_gru[i, j] > 80 else 'black', fontweight='bold')

# 2. ROC Curves
fpr_cnn, tpr_cnn, _ = roc_curve(sub_y, cnn_probs)
fpr_gru, tpr_gru, _ = roc_curve(sub_y, gru_probs)
auc_cnn = roc_auc_score(sub_y, cnn_probs)
auc_gru = roc_auc_score(sub_y, gru_probs)

axes[1].plot(fpr_cnn, tpr_cnn, label=f'CNN Baseline (AUC = {auc_cnn:.4f})', color='#2b5c8f', lw=2.2)
axes[1].plot(fpr_gru, tpr_gru, label=f'CNN-GRU Hybrid (AUC = {auc_gru:.4f})', color='#d95f02', lw=2.2)
axes[1].plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random Guess')
axes[1].set_title('Receiver Operating Characteristic (ROC) Curves', fontsize=12, fontweight='bold')
axes[1].set_xlabel('False Positive Rate (FPR)')
axes[1].set_ylabel('True Positive Rate (Recall)')
axes[1].legend(loc='lower right')
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


### 6. Error Matrix & Discriminative Separability
The ROC curves show that both models offer substantial discriminative capability over random guessing (AUC > 0.81). The confusion matrix reveals balanced precision and recall for CNN-GRU, with 134 true positives, 134 true negatives, 46 false positives, and 46 false negatives.


In [ ]:
with open(os.path.join(PROJECT_DIR, 'outputs', 'configs', 'threshold_config.json'), 'r') as f:
    cfg = json.load(f)

tuned_df = pd.read_csv(os.path.join(PROJECT_DIR, 'outputs', 'metrics', 'tuned_threshold_test_evaluation.csv'))
print('=== Test Performance at Validation-Tuned Operating Thresholds ===')
display(tuned_df[display_cols[:-1]].style.format({
    'accuracy': '{:.2%}', 'precision': '{:.2%}', 'recall': '{:.2%}',
    'f1': '{:.2%}', 'roc_auc': '{:.4f}', 'false_alarm_rate_fpr': '{:.2%}',
    'missed_fault_rate_fnr': '{:.2%}'
}))


### 7. Validation Threshold Engineering
By selecting a threshold constrained to **FPR $\le$ 15% on validation data** ($T = 0.75$ for CNN-GRU):
- **False Alarm Rate (FPR)** drops dramatically from **25.56% to 12.22%**.
- **Precision** increases from **74.44% to 83.70%**.
- **Accuracy** increases to **75.28%**.
- **ROC-AUC** remains unchanged at **0.8149**, proving that threshold selection adjusts the decision boundary along the fixed ROC manifold without data leakage.


In [ ]:
noise_df = pd.read_csv(os.path.join(PROJECT_DIR, 'outputs', 'metrics', 'noise_condition_evaluation.csv'))
machine_df = pd.read_csv(os.path.join(PROJECT_DIR, 'outputs', 'metrics', 'machine_type_evaluation.csv'))

print('=== Test Set Metrics by Ambient Noise Condition ===')
display(noise_df.style.format({
    'accuracy': '{:.2%}', 'precision': '{:.2%}', 'recall': '{:.2%}',
    'f1': '{:.2%}', 'roc_auc': '{:.4f}', 'false_alarm_rate_fpr': '{:.2%}',
    'missed_fault_rate_fnr': '{:.2%}'
}))

print('\n=== Test Set Metrics by Machine Type ===')
display(machine_df.style.format({
    'accuracy': '{:.2%}', 'precision': '{:.2%}', 'recall': '{:.2%}',
    'f1': '{:.2%}', 'roc_auc': '{:.4f}', 'false_alarm_rate_fpr': '{:.2%}',
    'missed_fault_rate_fnr': '{:.2%}'
}))


### 8. Subpopulation & Environmental Robustness Analysis
1. **Noise Condition Analysis:** As background factory noise intensifies from +6 dB to -6 dB, ROC-AUC decreases from ~0.88 to ~0.74–0.78. Under the harshest condition (-6 dB), CNN-GRU shows greater noise robustness, detecting **75.44%** of faults vs **61.40%** for CNN.
2. **Machine Type Analysis:** 
   - **Sliders:** Highest performance (ROC-AUC 0.9525), as guide friction creates continuous acoustic wear harmonics.
   - **Fans & Pumps:** Robust performance (ROC-AUC 0.76–0.88), benefiting from recurrent modeling of periodic unbalance.
   - **Valves:** Lowest performance (ROC-AUC ~0.66–0.70), due to transient impulsive clicks easily obscured by background factory noise.


In [ ]:
from src.audio_processing import preprocess_audio_file, prepare_input_tensor

sample_path = os.path.join(PROJECT_DIR, '6_dB_fan', 'fan', 'id_00', 'abnormal', '00000000.wav')

log_mel = preprocess_audio_file(sample_path)
input_tensor = prepare_input_tensor(log_mel, device='cpu')

with torch.no_grad():
    cnn_prob = torch.sigmoid(cnn(input_tensor)).item()
    gru_prob = torch.sigmoid(cnn_gru(input_tensor)).item()

print(f'Sample File: {os.path.basename(sample_path)}')
print(f'CNN Abnormal Probability: {cnn_prob*100:.2f}% -> Class: {"ABNORMAL" if cnn_prob >= 0.5 else "NORMAL"}')
print(f'CNN-GRU Abnormal Probability: {gru_prob*100:.2f}% -> Class: {"ABNORMAL" if gru_prob >= 0.5 else "NORMAL"}')


### 9. Conclusion and Capstone Summary
This capstone successfully establishes an automated acoustic predictive maintenance screening system. The CNN-GRU hybrid demonstrates tangible benefits over the CNN baseline in fault sensitivity (+6.11% recall) and noise resilience while maintaining low CPU latency (2.35 ms). 

Validation threshold tuning effectively lowers false alarms to 12.22% to satisfy factory operations. Future work will leverage GPU infrastructure to expand training across the full 50,000+ recording corpus and explore wavelet transforms for transient valve monitoring.
